# Zadanie 3 – Rekurencyjny współczynnik dwumianowy

## Czego wymaga zadanie?

Zdefiniować rekurencyjną funkcję `binomial_rek(n, k)` obliczającą **współczynnik dwumianowy** (symbol Newtona):

$$\binom{n}{k} = \frac{n!}{k!\,(n-k)!}, \quad 0 \le k \le n$$

**Ale bez liczenia silni!** Zamiast tego korzystamy z rekurencyjnej zależności (trójkąt Pascala):

$$\binom{n}{k} = \binom{n-1}{k-1} + \binom{n-1}{k}$$

z warunkami bazowymi:

$$\binom{n}{0} = 1 \qquad \binom{n}{n} = 1$$

### Dlaczego taki sposób?

Silnia rośnie **bardzo szybko** – dla dużych `n` `n!` przekracza zakres typowych typów całkowitych.  
Trójkąt Pascala pozwala obliczyć wynik, **operując tylko na mniejszych wartościach** bez ryzyka przepełnienia  
(Python obsługuje duże liczby całkowite natywnie, ale zasada dydaktyczna jest ważna).

> **Uwaga z treści zadania:** algorytm rekurencyjny oblicza aż $2^n - 1$ wyrazów,  
> co oznacza **wykładniczą złożoność** – dla dużych `n` jest wolny (patrz sekcja o złożoności poniżej).

## Implementacja funkcji

In [ ]:
def binomial_rek(n, k):
    """
    Rekurencyjne obliczanie symbolu Newtona C(n, k) przez trójkąt Pascala.
    Warunki bazowe: C(n,0) = 1 i C(n,n) = 1.
    """
    # Warunek bazowy 1: k=0 → C(n, 0) = 1
    # Matematycznie: wybieramy 0 elementów z n na dokładnie 1 sposób.
    if k == 0:
        return 1

    # Warunek bazowy 2: k=n → C(n, n) = 1
    # Matematycznie: wybieramy wszystkie n elementów – tylko 1 sposób.
    if k == n:
        return 1

    # Przypadek rekurencyjny: trójkąt Pascala
    # C(n, k) = C(n-1, k-1) + C(n-1, k)
    # Rozkładamy problem na dwa mniejsze – wyraz z lewej i prawej gałęzi drzewa rekurencji.
    return binomial_rek(n - 1, k - 1) + binomial_rek(n - 1, k)

### Jak działa rekurencja – przykład `C(4, 2)`

```
C(4,2)
├── C(3,1)
│   ├── C(2,0) = 1   ← warunek bazowy
│   └── C(2,1)
│       ├── C(1,0) = 1   ← warunek bazowy
│       └── C(1,1) = 1   ← warunek bazowy
└── C(3,2)
    ├── C(2,1)
    │   ├── C(1,0) = 1   ← warunek bazowy
    │   └── C(1,1) = 1   ← warunek bazowy
    └── C(2,2) = 1   ← warunek bazowy
```

Wynik: `1 + 1 + 1 + 1 + 1 + 1 = 6` ✓  
Sprawdzenie: $C(4,2) = \frac{4!}{2! \cdot 2!} = \frac{24}{4} = 6$

## Testy podstawowe

In [ ]:
# Przypadki brzegowe – warunki bazowe
print("Warunki bazowe:")
print(f"  C(5, 0) = {binomial_rek(5, 0)}   (oczekiwane: 1)")
print(f"  C(5, 5) = {binomial_rek(5, 5)}   (oczekiwane: 1)")
print(f"  C(1, 0) = {binomial_rek(1, 0)}   (oczekiwane: 1)")
print(f"  C(1, 1) = {binomial_rek(1, 1)}   (oczekiwane: 1)")

In [ ]:
# Znane wartości do weryfikacji
print("Weryfikacja wybranych wartości:")
test_cases = [
    (4, 2, 6),     # C(4,2) = 6
    (5, 2, 10),    # C(5,2) = 10
    (6, 3, 20),    # C(6,3) = 20
    (10, 5, 252),  # C(10,5) = 252
    (10, 0, 1),    # C(10,0) = 1
    (10, 10, 1),   # C(10,10) = 1
]

for n, k, expected in test_cases:
    result = binomial_rek(n, k)
    status = "OK" if result == expected else "BLAD"
    print(f"  C({n:2d}, {k:2d}) = {result:5d}   (oczekiwane: {expected:5d})  [{status}]")

## Wizualizacja – wiersz trójkąta Pascala

Dla danego `n` wypisujemy cały wiersz trójkąta Pascala:  
$C(n,0), C(n,1), \ldots, C(n,n)$

In [ ]:
def wypisz_wiersz_pascala(n):
    """Wypisuje n-ty wiersz trójkąta Pascala."""
    wiersz = [binomial_rek(n, k) for k in range(n + 1)]
    print(f"n={n}: {wiersz}")
    return wiersz

print("Trójkąt Pascala (wiersze 0–8):")
for n in range(9):
    wiersz = [binomial_rek(n, k) for k in range(n + 1)]
    # Wyśrodkowanie dla ładnego wydruku
    print("  " + " ".join(f"{v:4d}" for v in wiersz))

## Złożoność obliczeniowa – licznik wywołań

Treść zadania ostrzega: algorytm oblicza aż $2^n - 1$ wyrazów dla $\binom{n}{k}$.  
Poniżej sprawdzamy to empirycznie – liczymy, ile razy funkcja jest wywołana.

In [ ]:
# Wersja z licznikiem wywołań
wywolania = [0]  # lista zamiast int, bo Python nie pozwala modyfikować
                  # zmiennej zewnętrznej z wnętrza funkcji zagnieżdżonej

def binomial_rek_z_licznikiem(n, k):
    wywolania[0] += 1  # każde wejście do funkcji = jedno wywołanie
    if k == 0 or k == n:
        return 1
    return binomial_rek_z_licznikiem(n - 1, k - 1) + binomial_rek_z_licznikiem(n - 1, k)

print(f"{'n':>4} {'k':>4} {'C(n,k)':>10} {'wywołań':>10} {'2^n - 1':>10}")
print("-" * 44)
for n in range(2, 16):
    k = n // 2  # najgorszy przypadek – środkowy element (największy)
    wywolania[0] = 0
    wynik = binomial_rek_z_licznikiem(n, k)
    print(f"{n:>4} {k:>4} {wynik:>10} {wywolania[0]:>10} {2**n - 1:>10}")

### Wniosek ze złożoności

Liczba wywołań dla $\binom{n}{\lfloor n/2 \rfloor}$ rośnie **wykładniczo** i zbliża się do $2^n - 1$.  

Dla $n = 30$ algorytm wykona ponad **miliard** wywołań – to czyni go niepraktycznym dla dużych wartości.  
W praktyce stosuje się **memoizację** (zapamiętywanie wyników pośrednich) lub obliczanie iteracyjne przez trójkąt Pascala.

Na tym laboratorium implementujemy wersję **czystą rekurencyjną** zgodnie z treścią zadania.

## Podsumowanie

| Element | Opis |
|---|---|
| **Funkcja** | `binomial_rek(n, k)` |
| **Warunki bazowe** | `k == 0` lub `k == n` → zwraca `1` |
| **Przypadek rekurencyjny** | `C(n-1, k-1) + C(n-1, k)` (trójkąt Pascala) |
| **Brak silni** | dzięki rozkładowi rekurencyjnemu |
| **Złożoność** | $O(2^n)$ – wykładnicza, do $2^n - 1$ wywołań |